In [7]:
import pandas as pd
import os
from cc import load_df, merge_left, limited_metrics, full_metrics

In [ ]:
#  Run name
RUN_NAME = "screening_run"  # set before running

#  Paths 

# Full-text screening labels for the train set (ground truth, all ELIGIBLE)
S_TRAIN_FULL_PATH  = "../data/labels/l0/train/L0) Screening.csv"

# Manual abstract-based annotations (two sheets)
ANNOTATED_PATH     = "../data/consistency-check-datasets/screening/annotated_screening_dataset.xlsx"
S_TRAIN_ABS_SHEET  = "train-sample"   # sample from train set
D_SAMPLE_ABS_SHEET = "search-sample"  # random sample from broader literature

# LLM screening on the train sample
S_TRAIN_LLM_PATH  = "../data/labels/eval/train-sample-1504_1/data/screening.xlsx"

# LLM screening on the random search sample
D_SAMPLE_LLM_PATH = "../data/labels/eval/sample-l0-250226-01/data/screening.xlsx"

# Results collection 
_results = []

#  Per-check path metadata — used by the export cell for traceability 
_check_sources = {
    "Check 1: Manual vs Full-text":  {
        "truth":    S_TRAIN_FULL_PATH,
        "estimate": f"{ANNOTATED_PATH} [{S_TRAIN_ABS_SHEET}]",
    },
    "Check 2: LLM vs Full-text":     {
        "truth":    S_TRAIN_FULL_PATH,
        "estimate": S_TRAIN_LLM_PATH,
    },
    "Check 3: Manual vs LLM":        {
        "truth":    f"{ANNOTATED_PATH} [{S_TRAIN_ABS_SHEET}]",
        "estimate": S_TRAIN_LLM_PATH,
    },
    "Check 4: Search sample vs LLM": {
        "truth":    f"{ANNOTATED_PATH} [{D_SAMPLE_ABS_SHEET}]",
        "estimate": D_SAMPLE_LLM_PATH,
    },
}

## Consistency Check #1
**Manual abstract annotations (`s_train_abs`) vs full-text train labels (`s_train_full`)**

Truth: `s_train_full` (all `ELIGIBLE`) &nbsp;|&nbsp; Estimate: `s_train_abs`

In [9]:
left  = load_df(ANNOTATED_PATH,    
                sheet=S_TRAIN_ABS_SHEET,  
                id_col="custom_id",          
                label_col="eligibility"
                )

right = load_df(S_TRAIN_FULL_PATH,
                id_col="UT (Unique WOS ID)", 
                label_col="eligbility")

m1 = merge_left(left, right)
summary = limited_metrics(m1["y_true"], m1["y_pred"])
display(summary)
_results.append(("Check 1: Manual vs Full-text", summary))

  Matched: 100


,Metric,Value
0,Records evaluated,100.00
1,Observed Agreement,0.83
2,PABAK,0.66
3,Recall (ELIGIBLE),0.83
4,False Negative Rate,0.17
5,TP,83.00
6,FN,17.00


## Consistency Check #2
**LLM screening (`s_train_llm`) vs full-text train labels (`s_train_full`)**

Truth: `s_train_full` (all `ELIGIBLE`) &nbsp;|&nbsp; Estimate: `s_train_llm`

In [10]:
left  = load_df(S_TRAIN_LLM_PATH,  
                id_col="custom_id",          
                label_col="pred_label")

right = load_df(S_TRAIN_FULL_PATH, 
                id_col="UT (Unique WOS ID)", 
                label_col="eligbility")

m2 = merge_left(left, right)
summary = limited_metrics(m2["y_true"], m2["y_pred"])
display(summary)
_results.append(("Check 2: LLM vs Full-text", summary))

  Matched: 100


,Metric,Value
0,Records evaluated,100.00
1,Observed Agreement,0.79
2,PABAK,0.58
3,Recall (ELIGIBLE),0.79
4,False Negative Rate,0.21
5,TP,79.00
6,FN,21.00


## Consistency Check #3
**Manual abstract annotations (`s_train_abs`) vs LLM screening (`s_train_llm`)**

Truth: `s_train_abs` (manual) &nbsp;|&nbsp; Estimate: `s_train_llm`

In [11]:
left  = load_df(S_TRAIN_LLM_PATH, 
                id_col="custom_id", 
                label_col="pred_label"
                )

right = load_df(ANNOTATED_PATH,   
                sheet=S_TRAIN_ABS_SHEET, 
                id_col="custom_id", 
                label_col="eligibility"
                )

m3 = merge_left(left, right)
summary = limited_metrics(m3["y_true"], m3["y_pred"])
display(summary)
_results.append(("Check 3: Manual vs LLM", summary))

  Matched: 100


,Metric,Value
0,Records evaluated,100.0000
1,Observed Agreement,0.9000
2,PABAK,0.8000
3,Recall (ELIGIBLE),0.9157
4,False Negative Rate,0.0843
5,TP,76.0000
6,FN,7.0000


## Consistency Check #4
**Manual annotations on search sample (`d_sample_abs`) vs LLM screening (`d_sample_llm`)**

Truth: `d_sample_abs` (manual, mixed labels) &nbsp;|&nbsp; Estimate: `d_sample_llm`

In [12]:
left  = load_df(D_SAMPLE_LLM_PATH, 
                id_col="custom_id", 
                label_col="pred_label"
                )

right = load_df(ANNOTATED_PATH,    
                sheet=D_SAMPLE_ABS_SHEET, 
                id_col="custom_id", 
                label_col="eligibility"
                )

m4 = merge_left(left, right)
summary = full_metrics(m4["y_true"], m4["y_pred"])
display(summary)
_results.append(("Check 4: Search sample vs LLM", summary))

  [9600 left-side rows not found in right — dropped]
  Matched: 400


,Metric,Value
0,Records evaluated,400.0000
1,Observed Agreement,0.8925
2,Cohen's Kappa,0.7458
3,PABAK,0.7850
4,TP,100.0000
5,FP,22.0000
6,FN,21.0000
7,TN,257.0000
8,Precision (ELIGIBLE),0.8197
9,Recall (ELIGIBLE),0.8264


# Output

In [ ]:
# --- Save results to xlsx ---
OUTPUT_DIR  = f"../data/labels/eval/consistency-check/screening/{RUN_NAME}"
OUTPUT_XLSX = f"{OUTPUT_DIR}/screening_results.xlsx"

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not _results:
    print("Nothing to save")
else:
    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        sheet = "Screening"
        row = 0
        for check_name, metrics_df in _results:
            pd.DataFrame([[f"── {check_name} ──"]]).to_excel(
                writer, sheet_name=sheet, startrow=row, index=False, header=False)
            row += 1
            src = _check_sources.get(check_name, {})
            if src:
                pd.DataFrame([
                    [f"Truth:    {src['truth']}"],
                    [f"Estimate: {src['estimate']}"],
                ]).to_excel(writer, sheet_name=sheet, startrow=row, index=False, header=False)
                row += 2
            metrics_df.to_excel(writer, sheet_name=sheet, startrow=row, index=False)
            row += len(metrics_df) + 3
    print(f"Saved - {OUTPUT_XLSX}")